# Tiny LLM — Ablation Study Notebook

Sequential Optuna experiments isolating individual hyperparameters.
Run in order; fill in each `BEST_*` variable before running the next experiment.

| # | Study | Varies | Fixed at |
|---|-------|--------|----------|
| 0 | `lr_search` | learning rate | baseline architecture |
| 1 | `norm_ablation` | norm_position {pre, post, pre_post} | BEST_LR |
| 2 | `activation_ablation` | activation {gelu, swiglu} + d_ff | BEST_LR, BEST_NORM |
| 3 | `layers_ablation` | n_layers | BEST_LR, BEST_NORM, BEST_ACT |
| 4 | `dmodel_ablation` | d_model (n_heads derived) | same |
| 5 | `dff_ablation` | d_ff | same |
| 6 | `heads_ablation` | n_heads | same |
| 7 | `decay_ablation` | weight_decay | same |
| 8 | `seed_stability` | random seed — measures training stability | all best params |


In [ ]:
import os, sys, getpass, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not Path("40m-tiny-llm").exists():
        token = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub token: ")
        subprocess.run(["git", "clone",
            f"https://{token}@github.com/marcbloech/40m-tiny-llm.git"], check=True)
    os.chdir("40m-tiny-llm")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna", "optuna-dashboard"], check=True)

_cwd = Path(os.getcwd()).resolve()
REPO_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "pyproject.toml").exists() and (p / "src").exists()),
    _cwd,
)
os.chdir(REPO_ROOT)
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

if IN_COLAB:
    import shutil
    GDRIVE_TOK = Path("/content/drive/MyDrive/tiny_llm/data/tokenized")
    DATA_DIR   = REPO_ROOT / "data" / "tokenized"
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for fname in ["train.bin", "val.bin"]:
        dst, src = DATA_DIR / fname, GDRIVE_TOK / fname
        if not dst.exists():
            if src.exists():
                print(f"Copying {fname}...")
                shutil.copy(src, dst)
            else:
                raise FileNotFoundError(f"{fname} not found at {src}")
else:
    DATA_DIR = REPO_ROOT / "data" / "tokenized"

TRAIN_BIN = DATA_DIR / "train.bin"
VAL_BIN   = DATA_DIR / "val.bin"
assert TRAIN_BIN.exists(), f"Missing: {TRAIN_BIN}"
assert VAL_BIN.exists(),   f"Missing: {VAL_BIN}"
print(f"Repo root : {REPO_ROOT}")
print(f"train.bin : {TRAIN_BIN.stat().st_size / 1e9:.2f} GB")
print(f"val.bin   : {VAL_BIN.stat().st_size / 1e9:.2f} GB")


In [ ]:
import gc, math, random, warnings
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from tiny_llm.model.transformer import GPTModel
from tiny_llm.data.dataset import MemmapDataset
from tiny_llm.training.optimizer import create_optimizer, create_scheduler

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)
print("Imports OK")


In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print(f"device : {device}")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU    : {props.name}  ({props.total_memory / 1e9:.1f} GB VRAM)")
print(f"AMP    : {USE_AMP}")


In [ ]:
if device.type == "cuda":
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram_gb >= 35:
        BATCH_SIZE, GRAD_ACCUM, EVAL_BATCH_SIZE = 32, 8, 32
    elif vram_gb >= 14:
        BATCH_SIZE, GRAD_ACCUM, EVAL_BATCH_SIZE = 16, 16, 32
    else:
        BATCH_SIZE, GRAD_ACCUM, EVAL_BATCH_SIZE = 8, 32, 16
elif device.type == "mps":
    BATCH_SIZE, GRAD_ACCUM, EVAL_BATCH_SIZE = 16, 16, 16
else:
    BATCH_SIZE, GRAD_ACCUM, EVAL_BATCH_SIZE = 4, 4, 4

EFF_TOK = BATCH_SIZE * GRAD_ACCUM * 1024
print(f"Batch: {BATCH_SIZE} x {GRAD_ACCUM} grad_accum = {BATCH_SIZE*GRAD_ACCUM} seqs/step -> {EFF_TOK/1e3:.0f}K tok/step")


In [ ]:
# ── Baseline ~17 M param model (GELU, RoPE, pre-norm, AdamW) ─────────────────
BASE_DMODEL  = 256
BASE_NHEADS  = BASE_DMODEL // 32   # 8 heads, head_dim=32
BASE_NLAYERS = 8
BASE_DFF     = 512
BASE_ACT     = "gelu"
BASE_NORM    = "pre"
BASE_WD      = 0.1

VOCAB_SIZE     = 50257
CONTEXT_LENGTH = 1024
POS_ENC        = "rope"
WEIGHT_TYING   = True
BIAS           = False
QK_NORM        = False
DROPOUT        = 0.0

# Training budget per trial
TRAIN_STEPS  = 500
WARMUP_STEPS = 50
EVAL_STEPS   = 40
REPORT_EVERY = 100

# Shared DB — separate study names keep experiments isolated
if IN_COLAB:
    GDRIVE_OPTUNA = Path("/content/drive/MyDrive/tiny_llm/optuna")
    GDRIVE_OPTUNA.mkdir(parents=True, exist_ok=True)
    DB_PATH = GDRIVE_OPTUNA / "tiny_llm_ablation.db"
else:
    DB_PATH = REPO_ROOT / "notebooks" / "tiny_llm_ablation.db"
STORAGE = f"sqlite:///{DB_PATH}"

def approx_param_count(d_model, n_layers, d_ff, activation="gelu"):
    ffn = 3 if activation == "swiglu" else 2
    emb = VOCAB_SIZE * d_model
    blk = 4 * d_model**2 + ffn * d_model * d_ff + 2 * d_model
    return emb + n_layers * blk + d_model

n_base = approx_param_count(BASE_DMODEL, BASE_NLAYERS, BASE_DFF, BASE_ACT)
print(f"Baseline : d={BASE_DMODEL} L={BASE_NLAYERS} h={BASE_NHEADS} d_ff={BASE_DFF} {BASE_ACT}")
print(f"Params   : {n_base / 1e6:.1f} M")
print(f"DB       : {DB_PATH}")


In [ ]:
def _next_batch(it, loader):
    try:
        return next(it)
    except StopIteration:
        return next(iter(loader))


@torch.no_grad()
def evaluate(model, n_steps):
    model.eval()
    losses = []
    for i, (x, y) in enumerate(val_loader):
        if i >= n_steps:
            break
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast(device.type, enabled=USE_AMP):
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


def run_trial(trial, model_cfg, lr, weight_decay):
    model = None
    try:
        model = GPTModel({
            "vocab_size": VOCAB_SIZE, "context_length": CONTEXT_LENGTH,
            "dropout": DROPOUT, "weight_tying": WEIGHT_TYING,
            "bias": BIAS, "qk_norm": QK_NORM,
            "positional_encoding": POS_ENC, "rope_theta": 10_000.0,
            **model_cfg,
        }).to(device)

        optimizer = create_optimizer(model, optimizer_name="adamw",
                                     learning_rate=lr, weight_decay=weight_decay,
                                     betas=(0.9, 0.95))
        scheduler = create_scheduler(optimizer, lr_schedule="cosine",
                                     warmup_steps=WARMUP_STEPS,
                                     max_steps=TRAIN_STEPS, min_lr_ratio=0.1)
        scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
        model.train()
        train_iter = iter(train_loader)
        optimizer.zero_grad(set_to_none=True)

        for step in range(TRAIN_STEPS):
            for _ in range(GRAD_ACCUM):
                x, y = _next_batch(train_iter, train_loader)
                x, y = x.to(device), y.to(device)
                with torch.amp.autocast(device.type, enabled=USE_AMP):
                    _, loss = model(x, y)
                scaler.scale(loss / GRAD_ACCUM).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            if (step + 1) % REPORT_EVERY == 0:
                val_loss = evaluate(model, EVAL_STEPS)
                trial.report(val_loss, step)
                if trial.should_prune():
                    raise optuna.TrialPruned()

        return evaluate(model, EVAL_STEPS)

    except optuna.TrialPruned:
        raise
    except Exception as e:
        if any(kw in str(e) for kw in ("CUDA", "cuda", "out of memory", "AcceleratorError")):
            raise optuna.TrialPruned()
        raise
    finally:
        if model is not None:
            try: del model
            except: pass
        gc.collect()
        if torch.cuda.is_available():
            try: torch.cuda.empty_cache()
            except: pass
            try: torch.cuda.reset_peak_memory_stats()
            except: pass


def show_results(study, varied_params=None):
    n_complete = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
    n_pruned   = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
    print(f"Completed: {n_complete}  |  Pruned: {n_pruned}")
    if n_complete == 0:
        print("No completed trials. Restart runtime + delete DB if CUDA errors occurred.")
        return
    rows = []
    for t in study.trials:
        if t.state != optuna.trial.TrialState.COMPLETE:
            continue
        row = {"#": t.number, "val_loss": round(t.value, 4), "ppl": round(math.exp(t.value), 1)}
        row.update(t.params)
        row.update({k: v for k, v in t.user_attrs.items() if k != "n_params"})
        rows.append(row)
    df = pd.DataFrame(rows).sort_values("val_loss").reset_index(drop=True)
    if varied_params:
        front = ["#", "val_loss", "ppl"] + [p for p in varied_params if p in df.columns]
        rest  = [c for c in df.columns if c not in front]
        df = df[front + rest]
    pd.set_option("display.float_format", "{:.4g}".format)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 220)
    print(df.to_string(index=False))
    best = study.best_trial
    print(f"\n{'='*50}")
    print(f"  BEST  trial #{best.number}   loss={best.value:.4f}   PPL={math.exp(best.value):.1f}")
    print(f"{'='*50}")
    for k, v in best.params.items():
        print(f"  {k:<20} {v}")


# ── Loaders built once, shared across all experiments ────────────────────────
train_ds = MemmapDataset(TRAIN_BIN, context_length=CONTEXT_LENGTH)
val_ds   = MemmapDataset(VAL_BIN,   context_length=CONTEXT_LENGTH)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=False, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=EVAL_BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=False, drop_last=False)
print(f"train: {len(train_ds):,} samples  |  val: {len(val_ds):,} samples")


---
## Experiment 0 — Learning Rate Search

Find the optimal LR for the baseline 17M architecture before varying anything else.

**Range:** `1e-5` (small reference point) → `5e-3`
Phase 2 results: `1e-3 > 6e-4 > 3e-4`, so optimum likely sits in `3e-4`–`3e-3`.
15 log-scale trials give dense coverage.

**Fixed:** d=256, L=8, h=8, d_ff=512, GELU, pre-norm, wd=0.1


In [ ]:
STUDY_0 = "tiny_llm_exp0_lr_search"

def objective_0(trial):
    lr = trial.suggest_float("lr", 1e-5, 5e-3, log=True)
    model_cfg = dict(n_layers=BASE_NLAYERS, n_heads=BASE_NHEADS, d_model=BASE_DMODEL,
                     d_ff=BASE_DFF, activation=BASE_ACT, norm_position=BASE_NORM)
    trial.set_user_attr("n_params_M",
        round(approx_param_count(BASE_DMODEL, BASE_NLAYERS, BASE_DFF, BASE_ACT) / 1e6, 1))
    return run_trial(trial, model_cfg, lr=lr, weight_decay=BASE_WD)

study_0 = optuna.create_study(
    direction="minimize", study_name=STUDY_0, storage=STORAGE, load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=REPORT_EVERY),
)
n_remaining = max(0, 15 - sum(1 for t in study_0.trials
                               if t.state == optuna.trial.TrialState.COMPLETE))
print(f"Exp 0 — LR search: {n_remaining} trials to run")
study_0.optimize(objective_0, n_trials=n_remaining, show_progress_bar=True)
show_results(study_0, varied_params=["lr"])


### Fill in `BEST_LR` from Exp 0 results above, then run the next cell

In [ ]:
BEST_LR = 1e-3   # <-- UPDATE: paste best lr from study_0 above
print(f"BEST_LR = {BEST_LR:.2e}")


---
## Experiment 1 — Norm Position

Compare three normalisation placements.

| Value | Formula | Notes |
|-------|---------|-------|
| `pre` | `x + sublayer(norm(x))` | GPT-2 / LLaMA standard; most stable |
| `post` | `norm(x + sublayer(x))` | Original Transformer; needs careful warmup |
| `pre_post` | norm before **and** after each sublayer | Sandwich norm; often improves post-norm stability |

9 trials = 3 per option.  **Fixed:** architecture + BEST_LR


In [ ]:
STUDY_1 = "tiny_llm_exp1_norm_ablation"

def objective_1(trial):
    norm_pos = trial.suggest_categorical("norm_position", ["pre", "post", "pre_post"])
    model_cfg = dict(n_layers=BASE_NLAYERS, n_heads=BASE_NHEADS, d_model=BASE_DMODEL,
                     d_ff=BASE_DFF, activation=BASE_ACT, norm_position=norm_pos)
    trial.set_user_attr("n_params_M",
        round(approx_param_count(BASE_DMODEL, BASE_NLAYERS, BASE_DFF, BASE_ACT) / 1e6, 1))
    return run_trial(trial, model_cfg, lr=BEST_LR, weight_decay=BASE_WD)

study_1 = optuna.create_study(
    direction="minimize", study_name=STUDY_1, storage=STORAGE, load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=False),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=REPORT_EVERY),
)
n_remaining = max(0, 9 - sum(1 for t in study_1.trials
                              if t.state == optuna.trial.TrialState.COMPLETE))
print(f"Exp 1 — Norm position: {n_remaining} trials to run")
study_1.optimize(objective_1, n_trials=n_remaining, show_progress_bar=True)
show_results(study_1, varied_params=["norm_position"])


### Fill in `BEST_NORM` from Exp 1 results above

In [ ]:
BEST_NORM = "pre"   # <-- UPDATE: "pre" / "post" / "pre_post"
print(f"BEST_NORM = {BEST_NORM!r}")


---
## Experiment 2 — Activation Function (GELU vs SwiGLU)

SwiGLU uses **3 FFN matrices** (gate · up · down); GELU uses **2** (fc1 · fc2).
To avoid conflating capacity with activation choice, `d_ff` is co-searched
so each activation can find its optimal width at a roughly equal parameter budget.

**Search space:**
- `activation`: {gelu, swiglu}
- `d_ff`: 320 – 640, step 32  (GELU baseline=512; param-equivalent SwiGLU ≈ 341)

12 trials = ~6 per activation with varied d_ff.  **Fixed:** architecture + BEST_LR + BEST_NORM


In [ ]:
STUDY_2 = "tiny_llm_exp2_activation_ablation"

def objective_2(trial):
    activation = trial.suggest_categorical("activation", ["gelu", "swiglu"])
    d_ff       = trial.suggest_int("d_ff", 320, 640, step=32)
    model_cfg  = dict(n_layers=BASE_NLAYERS, n_heads=BASE_NHEADS, d_model=BASE_DMODEL,
                      d_ff=d_ff, activation=activation, norm_position=BEST_NORM)
    trial.set_user_attr("n_params_M",
        round(approx_param_count(BASE_DMODEL, BASE_NLAYERS, d_ff, activation) / 1e6, 1))
    return run_trial(trial, model_cfg, lr=BEST_LR, weight_decay=BASE_WD)

study_2 = optuna.create_study(
    direction="minimize", study_name=STUDY_2, storage=STORAGE, load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=4, n_warmup_steps=REPORT_EVERY),
)
n_remaining = max(0, 12 - sum(1 for t in study_2.trials
                               if t.state == optuna.trial.TrialState.COMPLETE))
print(f"Exp 2 — Activation: {n_remaining} trials to run")
study_2.optimize(objective_2, n_trials=n_remaining, show_progress_bar=True)
show_results(study_2, varied_params=["activation", "d_ff", "n_params_M"])


### Fill in `BEST_ACT` and `BEST_DFF` from Exp 2 results above

In [ ]:
BEST_ACT = "gelu"   # <-- UPDATE: "gelu" / "swiglu"
BEST_DFF  = 512     # <-- UPDATE: best d_ff from study_2

print(f"BEST_ACT = {BEST_ACT!r}")
print(f"BEST_DFF  = {BEST_DFF}")


---
## Architecture Ablations — Experiments 3–7

Each experiment varies **one parameter**; all others are fixed at the best values
found in Exps 0–2 plus the baseline architecture.


In [ ]:
# Run this cell before starting Exp 3
ARCH_BASE = dict(
    d_model       = BASE_DMODEL,
    n_heads       = BASE_NHEADS,
    n_layers      = BASE_NLAYERS,
    d_ff          = BEST_DFF,
    activation    = BEST_ACT,
    norm_position = BEST_NORM,
)
_n = approx_param_count(ARCH_BASE["d_model"], ARCH_BASE["n_layers"],
                         ARCH_BASE["d_ff"],    ARCH_BASE["activation"])
print("Architecture ablation base:")
for k, v in ARCH_BASE.items():
    print(f"  {k:<15} {v}")
print(f"  {'n_params':<15} {_n/1e6:.1f} M")
print(f"  {'lr':<15} {BEST_LR:.2e}  weight_decay={BASE_WD}")


---
## Experiment 3 — Number of Layers

Tests: 4, 6, 8 (baseline), 10, 12, 14 — 12 trials (2 repeats per depth).
d_model=256 is fixed, so deeper = more params.


In [ ]:
STUDY_3 = "tiny_llm_exp3_layers_ablation"

def objective_3(trial):
    n_layers = trial.suggest_categorical("n_layers", [4, 6, 8, 10, 12, 14])
    model_cfg = {**ARCH_BASE, "n_layers": n_layers}
    trial.set_user_attr("n_params_M",
        round(approx_param_count(ARCH_BASE["d_model"], n_layers,
                                  ARCH_BASE["d_ff"], ARCH_BASE["activation"]) / 1e6, 1))
    return run_trial(trial, model_cfg, lr=BEST_LR, weight_decay=BASE_WD)

study_3 = optuna.create_study(
    direction="minimize", study_name=STUDY_3, storage=STORAGE, load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=False),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=REPORT_EVERY),
)
n_remaining = max(0, 12 - sum(1 for t in study_3.trials
                               if t.state == optuna.trial.TrialState.COMPLETE))
print(f"Exp 3 — n_layers: {n_remaining} trials to run")
study_3.optimize(objective_3, n_trials=n_remaining, show_progress_bar=True)
show_results(study_3, varied_params=["n_layers", "n_params_M"])


### Fill in `BEST_LAYERS` from Exp 3 results above

In [ ]:
BEST_LAYERS = 8   # <-- UPDATE from study_3
print(f"BEST_LAYERS = {BEST_LAYERS}")


---
## Experiment 4 — Model Width (d_model)

`n_heads` is derived as `d_model // 32` to keep head_dim=32 constant across all widths.

Tests: 128, 192, 256 (baseline), 320, 384 — 10 trials.


In [ ]:
STUDY_4 = "tiny_llm_exp4_dmodel_ablation"

def objective_4(trial):
    d_model = trial.suggest_categorical("d_model", [128, 192, 256, 320, 384])
    n_heads = d_model // 32
    model_cfg = {**ARCH_BASE, "d_model": d_model, "n_heads": n_heads,
                 "n_layers": BEST_LAYERS}
    trial.set_user_attr("n_heads", n_heads)
    trial.set_user_attr("n_params_M",
        round(approx_param_count(d_model, BEST_LAYERS,
                                  ARCH_BASE["d_ff"], ARCH_BASE["activation"]) / 1e6, 1))
    return run_trial(trial, model_cfg, lr=BEST_LR, weight_decay=BASE_WD)

study_4 = optuna.create_study(
    direction="minimize", study_name=STUDY_4, storage=STORAGE, load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=False),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=REPORT_EVERY),
)
n_remaining = max(0, 10 - sum(1 for t in study_4.trials
                               if t.state == optuna.trial.TrialState.COMPLETE))
print(f"Exp 4 — d_model: {n_remaining} trials to run")
study_4.optimize(objective_4, n_trials=n_remaining, show_progress_bar=True)
show_results(study_4, varied_params=["d_model", "n_heads", "n_params_M"])


### Fill in `BEST_DMODEL` from Exp 4 results above

In [ ]:
BEST_DMODEL = 256   # <-- UPDATE from study_4
print(f"BEST_DMODEL = {BEST_DMODEL}  (n_heads = {BEST_DMODEL // 32}, head_dim = 32)")


---
## Experiment 5 — Feed-Forward Dimension (d_ff)

Pure d_ff sweep with the best activation and model width now fixed.
Range 256–1024, step 64 — 12 trials.


In [ ]:
STUDY_5 = "tiny_llm_exp5_dff_ablation"

def objective_5(trial):
    d_ff    = trial.suggest_int("d_ff", 256, 1024, step=64)
    n_heads = BEST_DMODEL // 32
    model_cfg = {**ARCH_BASE, "d_model": BEST_DMODEL, "n_heads": n_heads,
                 "n_layers": BEST_LAYERS, "d_ff": d_ff, "activation": BEST_ACT}
    trial.set_user_attr("n_params_M",
        round(approx_param_count(BEST_DMODEL, BEST_LAYERS, d_ff, BEST_ACT) / 1e6, 1))
    return run_trial(trial, model_cfg, lr=BEST_LR, weight_decay=BASE_WD)

study_5 = optuna.create_study(
    direction="minimize", study_name=STUDY_5, storage=STORAGE, load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=4, n_warmup_steps=REPORT_EVERY),
)
n_remaining = max(0, 12 - sum(1 for t in study_5.trials
                               if t.state == optuna.trial.TrialState.COMPLETE))
print(f"Exp 5 — d_ff: {n_remaining} trials to run")
study_5.optimize(objective_5, n_trials=n_remaining, show_progress_bar=True)
show_results(study_5, varied_params=["d_ff", "n_params_M"])


### Fill in `BEST_DFF2` from Exp 5 results above

In [ ]:
BEST_DFF2 = 512   # <-- UPDATE from study_5
print(f"BEST_DFF2 = {BEST_DFF2}")


---
## Experiment 6 — Number of Attention Heads

d_model is fixed at BEST_DMODEL; varying n_heads changes **head_dim = d_model / n_heads**.
Parameter count is unaffected (total Q/K/V projection = 3 × d_model² regardless of heads).

| n_heads | head_dim (if d_model=256) |
|---------|--------------------------|
| 4 | 64 |
| 8 | 32 (baseline) |
| 16 | 16 |
| 32 | 8 |

8 trials = 2 per option.


In [ ]:
STUDY_6 = "tiny_llm_exp6_heads_ablation"

_valid_heads = [h for h in [4, 8, 16, 32] if BEST_DMODEL % h == 0]

def objective_6(trial):
    n_heads = trial.suggest_categorical("n_heads", _valid_heads)
    trial.set_user_attr("head_dim", BEST_DMODEL // n_heads)
    model_cfg = {**ARCH_BASE, "d_model": BEST_DMODEL, "n_heads": n_heads,
                 "n_layers": BEST_LAYERS, "d_ff": BEST_DFF2, "activation": BEST_ACT}
    trial.set_user_attr("n_params_M",
        round(approx_param_count(BEST_DMODEL, BEST_LAYERS, BEST_DFF2, BEST_ACT) / 1e6, 1))
    return run_trial(trial, model_cfg, lr=BEST_LR, weight_decay=BASE_WD)

study_6 = optuna.create_study(
    direction="minimize", study_name=STUDY_6, storage=STORAGE, load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=False),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=REPORT_EVERY),
)
n_remaining = max(0, 8 - sum(1 for t in study_6.trials
                              if t.state == optuna.trial.TrialState.COMPLETE))
print(f"Exp 6 — n_heads: {n_remaining} trials to run")
study_6.optimize(objective_6, n_trials=n_remaining, show_progress_bar=True)
show_results(study_6, varied_params=["n_heads", "head_dim"])


### Fill in `BEST_NHEADS` from Exp 6 results above

In [ ]:
BEST_NHEADS = 8   # <-- UPDATE from study_6
print(f"BEST_NHEADS = {BEST_NHEADS}  (head_dim = {BEST_DMODEL // BEST_NHEADS})")


---
## Experiment 7 — Weight Decay

AdamW weight decay was fixed at 0.1 throughout all prior experiments and never swept.
Range: 0.0 (no regularisation) – 0.3 — 10 trials.


In [ ]:
STUDY_7 = "tiny_llm_exp7_decay_ablation"

def objective_7(trial):
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.3)
    model_cfg = {**ARCH_BASE, "d_model": BEST_DMODEL, "n_heads": BEST_NHEADS,
                 "n_layers": BEST_LAYERS, "d_ff": BEST_DFF2, "activation": BEST_ACT}
    trial.set_user_attr("n_params_M",
        round(approx_param_count(BEST_DMODEL, BEST_LAYERS, BEST_DFF2, BEST_ACT) / 1e6, 1))
    return run_trial(trial, model_cfg, lr=BEST_LR, weight_decay=weight_decay)

study_7 = optuna.create_study(
    direction="minimize", study_name=STUDY_7, storage=STORAGE, load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=4, n_warmup_steps=REPORT_EVERY),
)
n_remaining = max(0, 10 - sum(1 for t in study_7.trials
                               if t.state == optuna.trial.TrialState.COMPLETE))
print(f"Exp 7 — weight_decay: {n_remaining} trials to run")
study_7.optimize(objective_7, n_trials=n_remaining, show_progress_bar=True)
show_results(study_7, varied_params=["weight_decay"])


### Fill in `BEST_WD` from Exp 7 results above

In [ ]:
BEST_WD = 0.1   # <-- UPDATE from study_7
print(f"BEST_WD = {BEST_WD}")


---
## Experiment 8 — Seed Stability

**Goal:** Quantify how much val loss varies across different random initialisations
using the fully-optimised architecture and hyperparameters.

- Low std (< 0.005) → single run is reliable
- Moderate std (0.005–0.02) → average 2–3 seeds for the final model
- High std (> 0.02) → initialisation-sensitive; must average seeds

All 10 seeds are run exactly once (GridSampler, no pruning).


In [ ]:
# Assemble final best config — review before running Exp 8
BEST_CONFIG = dict(
    d_model       = BEST_DMODEL,
    n_heads       = BEST_NHEADS,
    n_layers      = BEST_LAYERS,
    d_ff          = BEST_DFF2,
    activation    = BEST_ACT,
    norm_position = BEST_NORM,
)
_n_best = approx_param_count(BEST_DMODEL, BEST_LAYERS, BEST_DFF2, BEST_ACT)
print("Final best config:")
for k, v in BEST_CONFIG.items():
    print(f"  {k:<15} {v}")
print(f"  {'n_params':<15} {_n_best/1e6:.1f} M")
print(f"  {'lr':<15} {BEST_LR:.2e}")
print(f"  {'weight_decay':<15} {BEST_WD}")


In [ ]:
STUDY_8  = "tiny_llm_exp8_seed_stability"
SEEDS_8  = [0, 1, 7, 13, 21, 42, 77, 99, 123, 256]

def objective_8(trial):
    seed = trial.suggest_categorical("seed", SEEDS_8)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    trial.set_user_attr("n_params_M",
        round(approx_param_count(BEST_DMODEL, BEST_LAYERS, BEST_DFF2, BEST_ACT) / 1e6, 1))
    return run_trial(trial, BEST_CONFIG, lr=BEST_LR, weight_decay=BEST_WD)

study_8 = optuna.create_study(
    direction="minimize", study_name=STUDY_8, storage=STORAGE, load_if_exists=True,
    sampler=optuna.samplers.GridSampler({"seed": SEEDS_8}),
    pruner=optuna.pruners.NopPruner(),
)
n_done = sum(1 for t in study_8.trials if t.state == optuna.trial.TrialState.COMPLETE)
n_remaining = max(0, len(SEEDS_8) - n_done)
print(f"Exp 8 — Seed stability: {n_remaining} trials to run ({n_done} already done)")
study_8.optimize(objective_8, n_trials=n_remaining, show_progress_bar=True)
show_results(study_8, varied_params=["seed"])


In [ ]:
n_complete = sum(1 for t in study_8.trials if t.state == optuna.trial.TrialState.COMPLETE)
if n_complete >= 2:
    completed = [t for t in study_8.trials if t.state == optuna.trial.TrialState.COMPLETE]
    losses     = [t.value for t in completed]
    seeds_done = [t.params["seed"] for t in completed]

    mu  = np.mean(losses)
    std = np.std(losses)
    rng = max(losses) - min(losses)

    print(f"Val loss across {n_complete} seeds:")
    print(f"  mean  = {mu:.4f}  (PPL {math.exp(mu):.1f})")
    print(f"  std   = {std:.4f}")
    print(f"  range = {rng:.4f}  (max - min)")
    print()
    if std < 0.005:
        verdict = "Very stable — single-seed training is reliable."
    elif std < 0.02:
        verdict = "Moderate variance — consider averaging 2-3 seeds."
    else:
        verdict = "High variance — training is init-sensitive. Use seed averaging."
    print(verdict)

    best_seed = seeds_done[int(np.argmin(losses))]
    print(f"\nBest seed : {best_seed}  (val_loss = {min(losses):.4f})")
    print(f"Use seed={best_seed} for the final full training run.")
else:
    print("Need at least 2 completed trials for stability analysis.")
